# Prepare data for finetuning

In [1]:
import os

# Virus DNA

In [2]:
DATA_DIR="phage_data/"

## Download datasets

### NCBI RefSeq bacteriophage genomes

First, install the NCBI Datasets CLI conda package as:

```
mm install -c conda-forge ncbi-datasets-cli
```

(replace `mm` with conda if you are using `conda` instead of micromamba)

In [3]:
folder_name="refseq/"

folder_path = os.path.join(DATA_DIR, folder_name)
os.makedirs(folder_path, exist_ok=True)

!datasets download virus genome taxon "Bacteriophage" --filename phages.zip --no-progressbar
!unzip phages.zip -d phages/
!mv phages/ncbi_dataset/data/* "{folder_path}"
!rm phages.zip
!rm -rf phages/

Archive:  phages.zip
  inflating: phages/README.md        
  inflating: phages/ncbi_dataset/data/data_report.jsonl  
  inflating: phages/ncbi_dataset/data/genomic.fna  
  inflating: phages/ncbi_dataset/data/virus_dataset.md  
  inflating: phages/ncbi_dataset/data/dataset_catalog.json  
  inflating: phages/md5sum.txt       


### PhagesDB

The downloaded link might change. If it does not work, go to `https://phagesdb.org/data/` and click on `Download Multifasta of All Actinobacteriophage Genomes`.

In [ ]:
folder_name="phagesdb/"

folder_path = os.path.join(DATA_DIR, folder_name)
os.makedirs(folder_path, exist_ok=True)

!wget -O "{os.path.join(folder_path, 'all_genomes.fasta')}" https://phagesdb.org/media/Actinobacteriophages-All.fasta

--2025-11-17 11:43:30--  https://phagesdb.org/media/Actinobacteriophages-All.fasta
Resolving phagesdb.org (phagesdb.org)... 45.79.158.136
Connecting to phagesdb.org (phagesdb.org)|45.79.158.136|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 348623718 (332M) [application/octet-stream]
Saving to: ‘phage_data/phagesdb/all_genomes.fasta’

phage_data/phagesdb 100%[===================>] 332.47M  27.8MB/s    in 13s     

2025-11-17 11:43:44 (26.5 MB/s) - ‘phage_data/phagesdb/all_genomes.fasta’ saved [348623718/348623718]



### NCBI GenBank Bacteriophages (DNA + RNA)

1. Go to [https://www.ncbi.nlm.nih.gov/nuccore](https://www.ncbi.nlm.nih.gov/nuccore) and search for `bacteriophage[Organism]`
2. Click on `Send to:` -> **File**, Format: **FASTA** -> `Create File`
3. Save it under *DATA_DIR/genbank*

In [13]:
folder_name = "genbank"

os.makedirs(os.path.join(DATA_DIR, folder_name), exist_ok=True)

## Data preprocessing

In [3]:
import os
from pathlib import Path
from Bio import SeqIO
from datasets import Dataset
from tqdm import tqdm
import pandas as pd

In [4]:
MAX_LEN = 2048
MIN_LEN = 50
VALID_CHARS = {"A", "C", "G", "T", "N"}

In [5]:
# Set to uppercase, remove whitespaces and replace any unknown character with N
def clean_sequence(seq: str) -> str:
    seq = seq.upper().replace(" ", "").replace("\n", "")
    cleaned = "".join(c if c in VALID_CHARS else "N" for c in seq)
    return cleaned

def load_all_sequences(fasta_dir: str, min_len: int) -> list[str]:
    """
    Loads all FASTA files inside `fasta_dir`.
    Returns a list of cleaned sequences.
    """
    sequences = []

    fasta_files = list(Path(fasta_dir).glob("**/*.fasta")) + \
                  list(Path(fasta_dir).glob("**/*.fa")) + \
                  list(Path(fasta_dir).glob("**/*.fna")) + \
                  list(Path(fasta_dir).glob("**/*.fas"))

    print(f"Found {len(fasta_files)} FASTA files.")

    for fasta_path in fasta_files:
        for record in tqdm(SeqIO.parse(str(fasta_path), "fasta"), desc=f"Loading {fasta_path}"):
            raw_seq = str(record.seq)
            cleaned = clean_sequence(raw_seq)

            if len(cleaned) >= min_len:
                sequences.append(cleaned)

    print(f"Loaded {len(sequences)} sequences.")
    return sequences

In [6]:
sequences = load_all_sequences(DATA_DIR, MIN_LEN)

Found 1 FASTA files.


Loading phage_data/refseq/genomic.fna: 13541it [00:22, 605.42it/s] 

Loaded 13304 sequences.


In [7]:
# Remove duplicates with hashing
def deduplicate_sequences(sequences: list[str]) -> list[str]:
    unique = {}
    for seq in tqdm(sequences, desc="Deduplicating..."):
        h = hash(seq)
        unique[h] = seq
    deduped = list(unique.values())
    print(f"Removed {len(sequences) - len(deduped)} sequences. {len(deduped)} remaining.")
    return deduped

sequences = deduplicate_sequences(sequences)

Deduplicating...: 100%|██████████| 13304/13304 [00:00<00:00, 44886.22it/s]

Removed 69 sequences. 13235 remaining.


In [8]:
def chunk_sequence(seq: str, max_len: int = MAX_LEN) -> list[str]:
    return [seq[i:i+max_len] for i in range(0, len(seq), max_len)]

# Divide long sequences
def chunk_sequences(sequences: list[str], max_len: int) -> list[str]:
    chunks = []

    for seq in sequences:
        if len(seq) <= max_len:
            chunks.append(seq)
        else:
            chunks.extend(chunk_sequence(seq, max_len))

    print(f"Total chunks: {len(chunks)}")
    return chunks

chunks = chunk_sequences(sequences, MAX_LEN)

Total chunks: 208752


In [9]:
hf_dataset = Dataset.from_dict({"sequence": chunks})
print(hf_dataset)

Dataset({
    features: ['sequence'],
    num_rows: 208752
})


In [10]:
hf_dataset.save_to_disk(os.path.join(DATA_DIR, "phage_clean_dataset"))

Saving the dataset (0/1 shards):   0%|          | 0/208752 [00:00<?, ? examples/s]

## Local dataset

In [12]:
def load_perphect_sequences(data_dir: str):
    bact = pd.read_csv(os.path.join(data_dir, "bacteria_df.csv"))["bacterium_sequence"].tolist()
    phag = pd.read_csv(os.path.join(data_dir, "phages_df.csv"))["phage_sequence"].tolist()

    return bact, phag

bact_sequences, phag_sequences = load_perphect_sequences("../data/perphect-data/all/")

In [13]:
bact_chunks = chunk_sequences(deduplicate_sequences(bact_sequences), MAX_LEN)
phag_chunks = chunk_sequences(deduplicate_sequences(phag_sequences), MAX_LEN)

Deduplicating...:   0%|          | 0/231 [00:00<?, ?it/s]

Deduplicating...: 100%|██████████| 231/231 [00:00<00:00, 378.63it/s]


Removed 5 sequences. 226 remaining.
Total chunks: 532519


Deduplicating...: 100%|██████████| 3539/3539 [00:00<00:00, 31100.36it/s]

Removed 339 sequences. 3200 remaining.


Total chunks: 105690


In [15]:
phages_hf_dataset = Dataset.from_dict({"sequence": phag_chunks})
print(phages_hf_dataset)

Dataset({
    features: ['sequence'],
    num_rows: 105690
})


In [16]:
bact_hf_dataset = Dataset.from_dict({"sequence": bact_chunks})
print(bact_hf_dataset)

Dataset({
    features: ['sequence'],
    num_rows: 532519
})


In [17]:
phages_hf_dataset.save_to_disk(os.path.join(DATA_DIR, "perphect_phage_dataset"))
bact_hf_dataset.save_to_disk(os.path.join(DATA_DIR, "perphect_bacteria_dataset"))

Saving the dataset (0/1 shards):   0%|          | 0/105690 [00:00<?, ? examples/s]

Saving the dataset (0/3 shards):   0%|          | 0/532519 [00:00<?, ? examples/s]